# Real-time Speech Recognition & Translation with Whisper Large V3

This notebook demonstrates how to use OpenAI's Whisper Large V3 model for real-time speech recognition and translation, integrated with Weights & Biases for MLOps.

## Features:
- State-of-the-art speech recognition with Whisper Large V3
- Real-time audio processing
- Multi-language translation
- W&B integration for experiment tracking

## 1. Setup and Installation

In [ ]:
# Install required packages
!pip install -q transformers torch accelerate sounddevice soundfile deep-translator wandb

In [ ]:
# Import libraries
import torch
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor, pipeline
from deep_translator import GoogleTranslator
import wandb
import numpy as np
import sounddevice as sd
import soundfile as sf

print("✅ All libraries imported successfully!")
print(f"🔥 PyTorch version: {torch.__version__}")
print(f"🎮 CUDA available: {torch.cuda.is_available()}")

## 2. Initialize Whisper Large V3 Model

In [ ]:
# Configuration
MODEL_ID = "openai/whisper-large-v3"
DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
TORCH_DTYPE = torch.float16 if torch.cuda.is_available() else torch.float32

print(f"🚀 Loading model on {DEVICE}...")

In [ ]:
# Load model and processor
model = AutoModelForSpeechSeq2Seq.from_pretrained(
    MODEL_ID,
    torch_dtype=TORCH_DTYPE,
    low_cpu_mem_usage=True,
    use_safetensors=True
)
model.to(DEVICE)

processor = AutoProcessor.from_pretrained(MODEL_ID)

print("✅ Model loaded successfully!")

In [ ]:
# Create pipeline
pipe = pipeline(
    "automatic-speech-recognition",
    model=model,
    tokenizer=processor.tokenizer,
    feature_extractor=processor.feature_extractor,
    max_new_tokens=128,
    chunk_length_s=30,
    batch_size=16,
    return_timestamps=True,
    torch_dtype=TORCH_DTYPE,
    device=DEVICE,
)

print("✅ Pipeline ready!")

## 3. Test with Sample Audio

In [ ]:
# Record a sample audio (5 seconds)
print("🎤 Recording for 5 seconds...")
print("Speak now!")

duration = 5  # seconds
sample_rate = 16000

audio_data = sd.rec(
    int(duration * sample_rate),
    samplerate=sample_rate,
    channels=1,
    dtype='float32'
)
sd.wait()

print("✅ Recording complete!")

# Flatten audio
audio_data = audio_data.flatten()

In [ ]:
# Transcribe audio
print("🔄 Transcribing...")

result = pipe(audio_data)
transcription = result["text"]

print("\n" + "="*80)
print("📝 Transcription:")
print(transcription)
print("="*80)

## 4. Translation

In [ ]:
# Setup translator
TARGET_LANGUAGE = "es"  # Change to your desired language code
translator = GoogleTranslator(target=TARGET_LANGUAGE)

print(f"✅ Translator ready for: {TARGET_LANGUAGE}")

In [ ]:
# Translate the transcription
if transcription.strip():
    translation = translator.translate(transcription)
    
    print("\n" + "="*80)
    print(f"🎤 Original: {transcription}")
    print(f"🌍 Translated ({TARGET_LANGUAGE}): {translation}")
    print("="*80)
else:
    print("⚠️ No speech detected in the recording")

## 5. Weights & Biases Integration

In [ ]:
# Initialize W&B (optional)
# Uncomment and set your API key

# import os
# os.environ["WANDB_API_KEY"] = "your_api_key_here"

# wandb.init(
#     project="whisper-realtime-translation",
#     config={
#         "model": MODEL_ID,
#         "device": DEVICE,
#         "target_language": TARGET_LANGUAGE,
#     }
# )

# print("✅ W&B initialized!")

In [ ]:
# Log to W&B
# if wandb.run:
#     wandb.log({
#         "transcription": transcription,
#         "translation": translation,
#         "transcription_length": len(transcription),
#         "audio_duration": duration,
#     })
#     print("✅ Logged to W&B")

## 6. Batch Processing Example

In [ ]:
# Process an audio file
# Replace with your audio file path
audio_file = "path/to/your/audio.wav"

# Uncomment to process
# result = pipe(audio_file)
# transcription = result["text"]
# translation = translator.translate(transcription)

# print(f"📝 Transcription: {transcription}")
# print(f"🌍 Translation: {translation}")

## 7. Real-time Continuous Processing

For production use, check out the `realtime_speech_translator.py` script which includes:
- Continuous audio streaming
- Chunked processing
- Queue-based architecture
- Better error handling
- Full W&B integration

In [ ]:
# Clean up
# if wandb.run:
#     wandb.finish()

print("✅ Session complete!")

## Next Steps

1. Try different target languages
2. Process your own audio files
3. Integrate with W&B for tracking
4. Use the production scripts for real-time processing

## Resources

- [Whisper Paper](https://arxiv.org/abs/2212.04356)
- [HuggingFace Model](https://huggingface.co/openai/whisper-large-v3)
- [W&B Documentation](https://docs.wandb.ai/)